# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maheen-armghan/flyrank-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [3]:
import os
if not os.path.exists("flyrank-internship"):
    !git clone https://github.com/maheen-armghan/flyrank-internship.git
os.chdir("flyrank-internship")
print("Now in:", os.getcwd())

Cloning into 'flyrank-internship'...
remote: Enumerating objects: 333, done.
remote: Counting objects: 100% (333/333), done.
remote: Compressing objects: 100% (149/149), done.
remote: Total 333 (delta 180), reused 293 (delta 156), pack-reused 0 (from 0)
Receiving objects: 100% (333/333), 1.92 MiB | 10.28 MiB/s, done.
Resolving deltas: 100% (180/180), done.
Now in: /content/flyrank-internship/flyrank-internship


In [4]:
!pip install duckdb -q
import duckdb
con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
print("DuckDB ready.")

DuckDB ready.


In [5]:
import os
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
os.environ["HF_TOKEN"] = hf_token

con.sql(f"""
CREATE OR REPLACE SECRET hf_secret (
    TYPE huggingface,
    TOKEN '{hf_token}'
);
""")

┌─────────┐
│ Success │
│ boolean │
├─────────┤
│ true    │
└─────────┘

**Unit of analysis: one row = one content page (`content_hash_id`), on one day (`report_date`)**, i.e. one row of `fact_content_daily_performance` represents a single page's performance metrics for a single day.

**Time window:** I'm developing against a mid-panel month, `month=2026-03`, per the guide's warning to never build label logic on the final `_sample` month (which is the actual outcome window, not a random sample).

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one, typing sentences here breaks Run All.
q = """
SELECT *
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
LIMIT 5
"""
sample = con.sql(q).df()
sample

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Feature fields** (knowable before the decision point): `gsc_impressions`, `gsc_clicks`, `gsc_avg_position`, derived `gsc_ctr`, `ga4_engaged_sessions` — real, same-day observed measurements.

**Label/proxy field:** for this demo, a same-day CTR-derived flag stands in as a placeholder target (`fake_label` in Section 3's leakage trap); a real capstone-grade label would instead use a strictly future window (e.g. decline over the next 30 days), never the current day.

**Context fields:** `client_hash_id`, `content_hash_id`, `report_date`, `gsc_data_available`, `ga4_data_available` — used for joins, filtering, and grouping, not as predictive features themselves.

**Excluded fields:** any FlyRank product decision output (`health_score`, `priority_score`, `action_type`) — not shipped in this dataset by design, to prevent circular results.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one, typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

**Claim 1 — grain:** one row = one page-day. Verified by checking that (content_hash_id, report_date) is unique within the month slice.

**Claim 2 — row count & date span:** the March 2026 slice contains [N] rows spanning [min_date] to [max_date].

**Claim 3 — availability:** filtering to rows with real GA4 availability (`ga4_data_available IS TRUE`) leaves [N2] of [N] rows — showing how much of the panel actually has engagement data versus search-only rows.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one, typing sentences here breaks Run All.
# Claim 1: grain check
q1 = """
SELECT COUNT(*) AS total_rows,
       COUNT(DISTINCT (content_hash_id, report_date)) AS unique_keys
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
"""
con.sql(q1).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,unique_keys
0,9841378,9841378


In [9]:
# Claim 2: row count and date span
q2 = """
SELECT COUNT(*) AS row_count, MIN(report_date) AS min_date, MAX(report_date) AS max_date
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
"""
con.sql(q2).df()

,row_count,min_date,max_date
0,9841378,2026-03-01,2026-03-31


In [10]:
# Claim 3: availability filter
q3 = """
SELECT COUNT(*) AS available_rows
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
WHERE ga4_data_available IS TRUE
"""
con.sql(q3).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,available_rows
0,413966


In [12]:
q_cols = """
DESCRIBE SELECT * FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
"""
con.sql(q_cols).df()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [13]:
feat_q = """
SELECT
  content_hash_id,
  report_date,
  gsc_impressions,                                    -- available when: observed the same day, before any refresh decision
  gsc_clicks,                                          -- available when: observed the same day
  gsc_avg_position,                                    -- available when: observed the same day
  CASE WHEN gsc_impressions > 0
       THEN gsc_clicks * 1.0 / gsc_impressions
       ELSE NULL END AS gsc_ctr,                       -- available when: derived same-day from clicks/impressions
  ga4_engaged_sessions                                 -- available when: observed the same day
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
WHERE gsc_data_available IS TRUE
LIMIT 1000
"""
features = con.sql(feat_q).df()
features.head()

,content_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_avg_position,gsc_ctr,ga4_engaged_sessions
0,content_b7e512995f79d5a6,2026-03-01,20,0,3.350000,0.000,<NA>
1,content_05597932fe4da067,2026-03-01,1,0,0.000000,0.000,<NA>
2,content_7a105f548d9c6916,2026-03-01,125,1,4.928000,0.008,<NA>
3,content_905aa32a0230694e,2026-03-01,7,0,4.000000,0.000,<NA>
4,content_a3ea9792f793ec72,2026-03-01,11,0,2.272727,0.000,<NA>


In [14]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

features["fake_label"] = (features["gsc_ctr"] > features["gsc_ctr"].median()).astype(int)

features["leaky_feature"] = features["gsc_ctr"] * 100

X_leaky = features[["gsc_impressions", "gsc_clicks", "gsc_avg_position", "leaky_feature"]].fillna(0)
y = features["fake_label"]
clf = DecisionTreeClassifier(max_depth=3).fit(X_leaky, y)
print("WITH leaky feature, AUC:", roc_auc_score(y, clf.predict_proba(X_leaky)[:,1]))

X_honest = features[["gsc_impressions", "gsc_clicks", "gsc_avg_position"]].fillna(0)
clf2 = DecisionTreeClassifier(max_depth=3).fit(X_honest, y)
print("WITHOUT leaky feature, AUC:", roc_auc_score(y, clf2.predict_proba(X_honest)[:,1]))

WITH leaky feature, AUC: 1.0
WITHOUT leaky feature, AUC: 1.0


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


This slice cannot tell me: (1) causal effects of any content change, only observational patterns; (2) anything about clients whose GA4 tracking hadn't started yet in this window (`ga4_data_available = FALSE` rows are search-only, not "zero engagement"); (3) the unbalanced panel means some clients have far more history than others, so any cross-client comparison must account for this rather than assume equal footing; (4) this month-only slice can't capture longer seasonal patterns, a full-year view would be needed for that.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.